# Data Preparation

## Transform nifti files to png(1024x1024)

In [32]:
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image

def process_nifti_to_png(img_path,mask_path, output_dir):
    """
    Load, preprocess, pad, upscale, and save a NIfTI file as PNG slices.
    
    Args:
        img_path (str): Path to the image NIfTI file.
        mask_path (str): Path to the mask NIfTI file.
        output_dir (str): Directory to save the PNG slices.
    """
    # Load and transpose the NIfTI file
    img = sitk.ReadImage(img_path)
    mask = sitk.ReadImage(mask_path)
    img_arr = sitk.GetArrayFromImage(img)  # Shape: (Depth, Height, Width)
    mask_arr = sitk.GetArrayFromImage(mask)  # Shape: (Depth, Height, Width)

    # Set the output dir for each 
    img_output_dir = output_dir+"/"+"images"
    mask_output_dir = output_dir+"/"+"masks"
    
    # Normalize the image (skip for masks)
    img_arr = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min()) * 255
    mask_arr = mask_arr.astype(np.uint8)
    img_arr = img_arr.astype(np.uint8)
    
    # Filter the image and mask arrays to keep only the non-empty pairs
    valid_indices = [i for i, mask in enumerate(mask_arr) if mask.max() != 0]
    filtered_images = img_arr[valid_indices]
    filtered_masks = mask_arr[valid_indices]

    # Process each slice
    for z in range(filtered_images.shape[0]):
        img_slice = filtered_images[z]
        mask_slice = filtered_masks[z]
        # Upscale the image
        upscaled_img = upscale_img(img_slice)
        upscaled_mask = upscale_img(mask_slice)
        # Save the image as PNG
        output_img = save_as_png(img_path, img_output_dir,z)
        output_mask = save_as_png(mask_path, mask_output_dir,z)
        upscaled_img.save(output_img)
        upscaled_mask.save(output_mask)



def upscale_img(slice_data):
    """
    Upscale the slice of data to 1024*1024    
    Args:
        slice_data (narray): numpy array of the slice of the image or mask.
    """
    # Convert to PIL Image for padding and resizing
    pil_img = Image.fromarray(slice_data)

    # Upscale to 1024x1024
    upscaled_img = pil_img.resize((1024, 1024), Image.Resampling.LANCZOS)
    
    return upscaled_img
        
def save_as_png(nii_path,output_dir,z):
    """
    Save the input in the output directory as png file.   
    Args:
        nii_path (string): path of the input
        output_dir (string): path of the output directory
        z (int): index of the slice
    """
    base_name = os.path.basename(nii_path).replace(".nii.gz", "")
    slice_name = f"{base_name}_slice{z:03d}.png"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, slice_name)

    return output_path
        

output_dir = "../Tumor_Seg"
process_nifti_to_png(
    img_path="../Brats_dataset_2024/imagesTr/BraTS-GLI-00002-000_0001.nii.gz",
    mask_path="../Brats_dataset_2024/labelsTr/BraTS-GLI-00002-000.nii.gz",
    output_dir=output_dir
)


In [30]:
import cv2
import random
import os

dir_path = "../Tumor_Seg"
imgs_dir = dir_path+"/images"
masks_dir = dir_path+"/masks"

for file in sorted(os.listdir(masks_dir)): # sort the files in the directory
    filename = os.fsdecode(file)
    print(file)
    

BraTS-GLI-00002-000_slice000.png
BraTS-GLI-00002-000_slice001.png
BraTS-GLI-00002-000_slice002.png
BraTS-GLI-00002-000_slice003.png
BraTS-GLI-00002-000_slice004.png
BraTS-GLI-00002-000_slice005.png
BraTS-GLI-00002-000_slice006.png
BraTS-GLI-00002-000_slice007.png
BraTS-GLI-00002-000_slice008.png
BraTS-GLI-00002-000_slice009.png
BraTS-GLI-00002-000_slice010.png
BraTS-GLI-00002-000_slice011.png
BraTS-GLI-00002-000_slice012.png
BraTS-GLI-00002-000_slice013.png
BraTS-GLI-00002-000_slice014.png
BraTS-GLI-00002-000_slice015.png
BraTS-GLI-00002-000_slice016.png
BraTS-GLI-00002-000_slice017.png
BraTS-GLI-00002-000_slice018.png
BraTS-GLI-00002-000_slice019.png
BraTS-GLI-00002-000_slice020.png
BraTS-GLI-00002-000_slice021.png
BraTS-GLI-00002-000_slice022.png
BraTS-GLI-00002-000_slice023.png
BraTS-GLI-00002-000_slice024.png
BraTS-GLI-00002-000_slice025.png
BraTS-GLI-00002-000_slice026.png
BraTS-GLI-00002-000_slice027.png
BraTS-GLI-00002-000_slice028.png
BraTS-GLI-00002-000_slice029.png
BraTS-GLI-

## Create CSV Files

In [33]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

images_dir = "../Tumor_Seg/images"
masks_dir = "../Tumor_Seg/masks"

# Get sorted pairs (ensures alignment)
imgs = [images_dir+"/"+f for f in sorted(os.listdir(images_dir))]
masks = [masks_dir+"/"+f for f in sorted(os.listdir(masks_dir))]
pairs = list(zip(imgs, masks))
# Split dataset (70-15-15)
train, test = train_test_split(pairs, test_size=0.3, random_state=42)
val, test = train_test_split(test, test_size=0.3, random_state=42)

# Save to CSV
def save_csv(pairs, csv_path):
    df = pd.DataFrame(pairs, columns=["image_path", "mask_path"])
    df.to_csv(csv_path, index=False, sep="\t")

save_csv(train, "../Tumor_Seg/train.csv")
save_csv(val, "../Tumor_Seg/val.csv")
save_csv(test, "../Tumor_Seg/test.csv")

#### Backup

In [ ]:
import os
import numpy as np
import SimpleITK as sitk
from PIL import Image

def process_nifti_to_png(nii_path, output_dir, is_mask=False):
    """
    Load, preprocess, pad, upscale, and save a NIfTI file as PNG slices.
    
    Args:
        nii_path (str): Path to the NIfTI file.
        output_dir (str): Directory to save the PNG slices.
        is_mask (bool): If True, treat the input as a mask (binary values).
    """
    # Load and transpose the NIfTI file
    img = sitk.ReadImage(nii_path)
    arr = sitk.GetArrayFromImage(img)  # Shape: (Depth, Height, Width)
    
    # Normalize the image (skip for masks)
    if not is_mask:
        arr = (arr - arr.min()) / (arr.max() - arr.min()) * 255
    arr = arr.astype(np.uint8)

    # Process each slice
    for z in range(arr.shape[0]):
        slice_data = arr[z]  # Extract slice
        
        # Convert to PIL Image for padding and resizing
        pil_img = Image.fromarray(slice_data)

        # Add padding to make it 256x256
        # padded_img = pil_img.resize((256, 256), Image.Resampling.NEAREST)

        # Upscale to 1024x1024
        upscaled_img = pil_img.resize((1024, 1024), Image.Resampling.LANCZOS)
        

        # Step 6: Save as PNG
        base_name = os.path.basename(nii_path).replace(".nii.gz", "")
        slice_name = f"{base_name}_slice{z:03d}.png"
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, slice_name)
        
        upscaled_img.save(output_path)

images_dir = "../Tumor_Seg/images"
masks_dir = "../Tumor_Seg/masks"

process_nifti_to_png(
    nii_path="../Brats_dataset_2024/imagesTr/BraTS-GLI-00000-000_0001.nii.gz",
    output_dir=images_dir,
    is_mask=False
)

process_nifti_to_png(
    nii_path="../Brats_dataset_2024/labelsTr/BraTS-GLI-00000-000.nii.gz",
    output_dir=masks_dir,
    is_mask=True
)